# Job Training Programme : Intent-to-Treat
### ITT , confidence interval and p value are computed from scratch in `self_module/stats.py` (the maths is in the comments there).

In [1]:
import sys
import pandas as pd
import plotly.express as px
from scipy import stats

sys.path.append('../..')
from self_module import diff_in_means

In [2]:
import plotly.io as pio
pio.templates["plotly_dark"].layout.paper_bgcolor = "black"
pio.templates["plotly_dark"].layout.plot_bgcolor = "black"
pio.templates.default = "plotly_dark"

In [3]:
df = pd.read_csv('../../data/jtpa_india_synthetic_40000.csv')       # downloaded in 01
z = df['assigned_to_training'].values
outcomes = ['employed_18mo' , 'employed_30mo' , 'earnings_18mo_inr' , 'earnings_30mo_inr']

## 2. Intent-to-Treat (ITT) Analysis
### ITT = mean outcome of those offered training - mean outcome of those not offered.
### The offer was random , so the two groups are alike in everything except the offer , and the gap is the effect of the offer.
### It is the effect of offering , not of training : 38% of those offered never enrolled.

In [4]:
itt = pd.DataFrame({o : diff_in_means(df[o].values , z) for o in outcomes}).T
itt.round(4)

,treated mean,control mean,diff,se,ci low,ci high,p
employed_18mo,0.4625,0.4313,0.0311,0.0053,0.0208,0.0415,0.0
employed_30mo,0.4693,0.4298,0.0396,0.0053,0.0293,0.0499,0.0
earnings_18mo_inr,58712.0861,52160.6390,6551.4471,773.3103,5035.7867,8067.1074,0.0
earnings_30mo_inr,76405.7337,67329.3182,9076.4154,1016.5630,7083.9886,11068.8423,0.0


In [5]:
emp = itt.loc[['employed_18mo' , 'employed_30mo']].reset_index(names='outcome')
fig = px.scatter(emp , x='diff' , y='outcome' , error_x=emp['ci high'] - emp['diff'] , title='ITT on employment (95% CI)')
fig.add_vline(x=0 , line_dash='dash')
fig.show()

In [6]:
earn = itt.loc[['earnings_18mo_inr' , 'earnings_30mo_inr']].reset_index(names='outcome')
fig = px.scatter(earn , x='diff' , y='outcome' , error_x=earn['ci high'] - earn['diff'] , title='ITT on earnings in INR (95% CI)')
fig.add_vline(x=0 , line_dash='dash')
fig.show()

### Employment : +3.1 points at 18 months , +4.0 points at 30 months.
### Earnings : +6,551 INR at 18 months , +9,076 INR at 30 months (95% CI 7,084 to 11,069) , 13.5% above the not offered group.
### Every p value is below 0.001 and no CI comes near 0. The effect also grows from 18 to 30 months.
### For policymakers : on average , every applicant offered a place earns about 9,000 INR more at 30 months , counting the 38% who never took it up.

### Simple scipy implementation

In [7]:
rows = {}
for o in outcomes:
    t = stats.ttest_ind(df[o][z == 1] , df[o][z == 0] , equal_var=False)
    ci = t.confidence_interval()
    rows[o] = {'ci low' : ci.low , 'ci high' : ci.high , 'p' : t.pvalue}
scipy_itt = pd.DataFrame(rows).T

### Our module vs scipy

In [8]:
pd.concat({'our module' : itt[['ci low' , 'ci high' , 'p']] , 'scipy' : scipy_itt} , axis=1).round(4)

our module                       scipy                 
                      ci low     ci high    p     ci low     ci high    p
employed_18mo         0.0208      0.0415  0.0     0.0208      0.0415  0.0
employed_30mo         0.0293      0.0499  0.0     0.0293      0.0499  0.0
earnings_18mo_inr  5035.7867   8067.1074  0.0  5035.7212   8067.1730  0.0
earnings_30mo_inr  7083.9886  11068.8423  0.0  7083.9026  11068.9283  0.0

### Same numbers. scipy uses the t distribution , which is the normal one at 13,000+ people per group.

## 4. Earnings , Employment and Heterogeneous Effects
### Zero earnings

In [9]:
zero = pd.DataFrame({
    '18 months' : (df['earnings_18mo_inr'] == 0).groupby(df['assigned_to_training']).mean() ,
    '30 months' : (df['earnings_30mo_inr'] == 0).groupby(df['assigned_to_training']).mean() ,
})
zero.index = ['not offered' , 'offered']
zero.round(3)

,18 months,30 months
not offered,0.569,0.570
offered,0.538,0.531


In [10]:
fig = px.histogram(df[df['earnings_30mo_inr'] > 0] , x='earnings_30mo_inr' , color='assigned_to_training' ,
                   barmode='overlay' , histnorm='probability' , nbins=60 , title='Earnings at 30 months , employed only')
fig.show()

### 57% of the not offered and 53% of the offered earn nothing at 30 months. Earnings are 0 exactly when not employed.
### So the earnings ITT mixes two things : more people working , and what each worker earns.
### Comparing earnings of the employed only is not a fair comparison : the offer changes who is employed , so the two employed groups are no longer alike.

### ITT by social category

In [11]:
rows = []
for group , d in df.groupby('social_category'):
    for o in outcomes:
        r = diff_in_means(d[o].values , d['assigned_to_training'].values)
        rows.append({'group' : group , 'outcome' : o , 'n' : len(d) , **r})
groups = pd.DataFrame(rows)
groups[['group' , 'outcome' , 'n' , 'diff' , 'ci low' , 'ci high' , 'p']].round(4)

,group,outcome,n,diff,ci low,ci high,p
0,General,employed_18mo,20000,0.0359,0.0213,0.0505,0.0000
1,General,employed_30mo,20000,0.0351,0.0205,0.0497,0.0000
2,General,earnings_18mo_inr,20000,6686.5968,4456.8552,8916.3383,0.0000
3,General,earnings_30mo_inr,20000,8328.9183,5413.9239,11243.9126,0.0000
4,OBC,employed_18mo,8800,0.0242,0.0021,0.0462,0.0315
5,OBC,employed_30mo,8800,0.0511,0.0291,0.0731,0.0000
6,OBC,earnings_18mo_inr,8800,6870.7789,3695.2222,10046.3356,0.0000
7,OBC,earnings_30mo_inr,8800,9735.7330,5480.0954,13991.3706,0.0000
8,SC,employed_18mo,7200,0.0399,0.0157,0.0641,0.0012
9,SC,employed_30mo,7200,0.0294,0.0051,0.0537,0.0177


In [12]:
g = groups[groups['outcome'] == 'employed_30mo']
fig = px.scatter(g , x='diff' , y='group' , error_x=g['ci high'] - g['diff'] , title='ITT on employment at 30 months by group (95% CI)')
fig.add_vline(x=0 , line_dash='dash')
fig.show()

In [13]:
g = groups[groups['outcome'] == 'earnings_30mo_inr']
fig = px.scatter(g , x='diff' , y='group' , error_x=g['ci high'] - g['diff'] , title='ITT on earnings at 30 months by group in INR (95% CI)')
fig.add_vline(x=0 , line_dash='dash')
fig.show()

### At 30 months every group gains in both employment and earnings , and every CI is above 0.
### ST has the largest point estimates (+6.3 points , +11,961 INR) but also the widest CIs , since it is the smallest group (4,000).
### At 18 months the ST effects are not significant (p = 0.36 and 0.07) , so for ST the gains show up later.
### The CIs overlap a lot , so this does not show that one group gains more than another.

### Multiple comparisons
### 4 groups x 4 outcomes = 16 tests. At 5% each , about 1 in 20 true zeros still shows p < 0.05 , so some "significant" subgroup results can be luck.
### Bonferroni : test each one at 0.05 / 16 , so the chance of any false positive stays under 5%.

In [14]:
groups['significant at 0.05'] = groups['p'] < 0.05
groups['significant after bonferroni'] = groups['p'] < 0.05 / len(groups)
groups.pivot(index='group' , columns='outcome' , values='significant after bonferroni')

outcome,earnings_18mo_inr,earnings_30mo_inr,employed_18mo,employed_30mo
group,,,,
General,True,True,True,True
OBC,True,True,False,True
SC,True,True,True,False
ST,False,True,False,True


### 12 of the 16 survive Bonferroni. All four 30-month earnings effects survive.
### Lost : OBC and SC employment at one horizon each , and both ST 18-month effects.